# Reproduce Solar Pitch Angle 

## Introduction

Try to reproduce the observed increase of surface brightness using a set of Solar angles between +50 to +50 

Author: Alejandro S. Borlaff - 13 Oct 2026 

## Imports

In [ ]:
import rosalia as rs
import numpy as np
import pandas as pd 
from tqdm import tqdm 
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord  # High-level coordinates
import matplotlib.pyplot as plt


#### Get a valid pointing at a certain distance from the Sun, which uses PA nominal (Sun perpendicular to the Solar Array)

- Step 0: Get the location of the Sun from Roman
- Step 1: Find any location (ra, dec) at X degrees from the Sun at a given t 
- Step 2: Determine the PA_WFI. 
- Step 3: Run rosalia/straylight 

In [ ]:
# Set the time of the exposure: 
date = Time("2026-10-01T00:00:00") # Date of the observation, in Astropy Time YYYY-MM-DDTHH:MM:SS format.

# Get the location of the Sun at the requested time. 
sso_bodies = rs.sso.get_SS0s_loc_magnitude(observer="Roman", MJD=date.mjd, lambda_ref=1.7*u.um, verbose=False)
sso_bodies

In [ ]:
sso_bodies["source_id_2"]

#### Get the pointing coordinates at the required distance to the Sun

In [ ]:
central_coords = SkyCoord(sso_bodies["RA"].iloc[0], sso_bodies["DEC"].iloc[0], frame="icrs", unit="deg")
radius = np.linspace(-50, 50, 15) + 90
position_angle = 0 # This is arbitrary. Let's do it in the North - South direction 
pointing_coords = central_coords.directional_offset_by(position_angle, separation=radius*u.degree)

print(pointing_coords)

plt.scatter(pointing_coords.ra, pointing_coords.dec)
plt.scatter(sso_bodies["RA"].iloc[0], sso_bodies["DEC"].iloc[0])
plt.scatter(sso_bodies["RA"].iloc[1], sso_bodies["DEC"].iloc[1])

In [ ]:
# Let's say that we want to observe a target at the following coordinates:
nominal_PAs = []
for i in range(len(pointing_coords)):
    nominal_PA = rs.telescopes.Roman.get_bestPA(ra=pointing_coords.ra[i].value, dec=pointing_coords.dec[i].value, mjd=date.mjd)
    nominal_PAs.append(nominal_PA)


#### Analyze the exposures with ROSALIA

In [ ]:
# This notebook demonstrates how to use ROSALIA to analyze stray light in a simple exposure. 

# Let's define the minimum parameters to generate a dummy Roman / WFI image
# To get some challenging environment, let's target the Pleiades.
i = 0 
import pandas as pd

for i in range(len(pointing_coords)):
    ra = pointing_coords.ra[i].value  # Right ascension, in degrees. 
    dec = pointing_coords.dec[i].value # Declination, in degrees.
    PA = nominal_PAs[i]  - 60 # Position angle, in degrees.



    star_catalog = pd.DataFrame({"ra": sso_bodies["RA"], 
                                "dec": sso_bodies["DEC"], 
                                "source_id": sso_bodies["source_id"], 
                                "cat_id": sso_bodies["source_id_2"],
                                "mag_lambda": sso_bodies["mag_custom"]})

    star_catalog= star_catalog[0:3]

    bandpass = "F146"
    exptime = 600 # Exposure time, in seconds.


    observer={"TELESCOP": "Roman/WFI", 
            "pointing": [ra, dec], 
            "FILTER":bandpass, 
            "PA_Y": PA, 
            "EXPSTART": date.mjd, 
            "EXPTIME": exptime}

    prefix = "SunPitch_" + str(i).zfill(3)

    # With this line we generate the exposure object. This is the first step in any ROSALIA analysis. 
    custom_roman_exposure = rs.core.exposure(observer=observer, prefix=prefix) 

    straylight_out = custom_roman_exposure.straylight(catalog=star_catalog)


## About this Notebook

**Author:** Alejandro S. Borlaff - NASA Ames Research Center (Code S) - a.s.borlaff@nasa.gov 

**Updated on** September 20, 2026. 